# XGBoost Baseline Training - XAUUSD H1 (ONNX Standard)
Notebook ini melatih model XGBoost dengan metode Walk-Forward Validation menggunakan target Threshold Return 3-Class. Hasil akhir akan di-ekspor murni menjadi format `.onnx` sesuai standarisasi mesin.

In [ ]:
!pip install onnx onnxmltools onnxconverter-common

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

# Pustaka khusus ONNX
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
import onnx

## 1. Load Data
Pastikan path disesuaikan jika dijalankan di Kaggle (misal: `/kaggle/input/dataset-nama/XAUUSD_H1_features.csv`)

In [ ]:
# Ubah string di bawah ini sesuai path dataset Anda di Kaggle
csv_path = 'XAUUSD_H1_features.csv' 
df = pd.read_csv(csv_path)

# Urutkan waktu secara kronologis
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)
print(f"Total data yang diload: {len(df)} baris")
df.head()

## 2. Pembuatan Target (Threshold Return 3-Class)

In [ ]:
N_BARS = 3
THRESHOLD = 0.0015 # Target return 0.15% untuk menghindari noise

# Hitung pergerakan harga 3 jam ke depan
df['future_return'] = (df['close'].shift(-N_BARS) - df['close']) / df['close']

# Pelabelan Threshold Return
conditions = [
    (df['future_return'] > THRESHOLD),
    (df['future_return'] < -THRESHOLD)
]
choices = [1, -1] # 1 untuk BUY, -1 untuk SELL
df['target'] = np.select(conditions, choices, default=0) # 0 untuk NEUTRAL/HOLD

print("Distribusi Label (Persentase):")
print(df['target'].value_counts(normalize=True))

## 3. Ekstraksi Fitur Tambahan & Pembersihan (Cleaning)

In [ ]:
# Menambahkan Temporal Window Features (Lag)
df['close_lag_1'] = df['close'].shift(1)
df['return_lag_1'] = (df['close'] - df['close_lag_1']) / df['close_lag_1']

# Hapus baris dengan nilai kosong (NaN) hasil operasi shift
df.dropna(inplace=True)

# Buang kolom yang tidak boleh dijadikan fitur (Kategori & Target)
fitur_kategori = ['time', 'future_return', 'target']
X = df.drop(columns=fitur_kategori)

# ONNX membutuhkan kepastian tipe data yang ketat (Float32 sangat disarankan)
X = X.astype(np.float32)
y = df['target']

# Persyaratan mutlak XGBoost: label wajib dimulai dari angka positif 0, 1, 2...
y_mapped = y.map({-1: 0, 0: 1, 1: 2})
print("Matriks Fitur (X) siap. Jumlah fitur:", X.shape[1])

## 4. Walk-Forward Validation & Training

In [ ]:
# Membagi data menjadi 5 lipatan bergeser (Time Series Split)
tscv = TimeSeriesSplit(n_splits=5)

# Inisialisasi Baseline XGBoost
model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

fold = 1
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_mapped.iloc[train_index], y_mapped.iloc[test_index]
    
    # Gunakan .values agar onnxmltools tidak error membaca string nama kolom
    model.fit(X_train.values, y_train)
    y_pred = model.predict(X_test.values)
    
    acc = accuracy_score(y_test, y_pred)
    print(f"Fold {fold} | Akurasi: {acc:.4f} | Testing Range: {df['time'].iloc[test_index[0]].date()} s/d {df['time'].iloc[test_index[-1]].date()}")
    fold += 1

print("\n--- Classification Report (Siklus Pengujian Terakhir) ---")
target_names = ['SELL (0)', 'NEUTRAL (1)', 'BUY (2)']
print(classification_report(y_test, y_pred, target_names=target_names))

## 5. Visualisasi Feature Importance & Export ke ONNX

In [ ]:
plt.figure(figsize=(10,6))
pd.Series(model.feature_importances_, index=X.columns).nlargest(15).sort_values().plot(kind='barh', color='steelblue')
plt.title("Top 15 Feature Importance (Penggerak Prediksi)")
plt.show()

# Konversi Model XGBoost ke Format Universal ONNX
print("Mengekspor model ke ONNX...")
initial_type = [('float_input', FloatTensorType([None, X.shape[1]]))] 
onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_type)

model_filename = 'xgboost_baseline_v1.onnx'
onnx.save(onnx_model, model_filename)
print(f"\nBerhasil menyimpan model ONNX ke: {model_filename}")
print("Silakan unduh file ini dan letakkan di folder quant-engine-v1/ml_models/ Anda.")